# Baseball Attendance

I think there are a few things that can be very interesting when looking at the attendance data for MLB games:
- Who the opponent is
- Date time information like day of the week, month, or game start time.
- If the team is in playoff contention

### The Data

Baseball attendance data is available in a couple different places. [Retorsheet](https://retrosheet.org/) is great for almost any stats we want, including attendance data. However, it misses a few of extra points I want to explore, like the number of game number or the number of games back in the standings a team is at any given point. These numbers could be calculated ourselves, but for simplicity I'm going to download the data from [Baseball Reference](https://www.baseball-reference.com/) instead.

[Here's a sample](https://www.baseball-reference.com/teams/WSN/2019-schedule-scores.shtml) of what a team's schedule looks like on Baseball Reference. There are several columns that will give more insight into attendance trends:

- `Date`
- `Tm`: The team the data was scraped from
- `Unnamed: 4`: `NaN` or `@` to denote the home team
- `Opp`: The opposing team
- `Rank`: Where the team is in the standings
- `GB`: How many games behind first place the team is in the standings
- `D/N`: Denote a day or night game
- `Attendance`: The number of tickets officially distributed (NOTE: this is *not* how many fans actually attending the game)
- `cLI`: Championship Leverage Index, metric for how important the game is in order to win the championship ([further reading](https://www.sports-reference.com/blog/2020/09/__trashed-2/))
- `Orig. Scheduled`: When the game was originally scheduled. Also provides other information if a game was played at a different location.

#### Baseball Reference

Baseball Reference is a great place to view data for baseball, however it can be a little challenging downloading data from them. First, they don't have any public datasets or an API available, so data needs to be scraped from their site. Second, scraping data from their site can be tedious because they rate limit requests to about 20 requests per minute.

I've included the full raw data in this repo (`raw_output.csv`) so you don't need to wait for the scraping to run, but if you want to download the data yourself you can run the cell below to fetch all of the data. To minimize waiting time, you can also adjust the `start_year` and `end_year`, to focus only on the years you'd like to view. Running the scraping script takes about 75 minutes.

In [3]:
from data_helper import DataHelper
# DataHelper().collect_data()
DataHelper(start_year=2025, end_year=2025).collect_data()

Using start_year=2025
Using end_year=2025
Created DataHelper!
Finished Diamondbacks
Finished Athletics
Finished Braves
Finished Orioles
Finished Red Sox
Finished Cubs
Finished White Sox
Finished Reds
Finished Guardians
Finished Rockies
Finished Tigers
Finished Astros
Finished Royals
Finished Angels
Finished Dodgers
Finished Marlins
Finished Brewers
Finished Twins
Finished Mets
Finished Yankees
Finished Phillies
Finished Pirates
Finished Padres
Finished Giants
Finished Mariners
Finished Cardinals
Finished Rays
Finished Rangers
Finished Blue Jays
Finished Nationals
Finished Collecting All Data!


### Load the Data

In [152]:
import pandas as pd

df = pd.read_csv('raw_output.csv')
# df.head()

### Cleaning the Data

First, there are several columns that can be removed:

In [153]:
df = df.drop(columns=['Gm#', 'W/L', 'W-L', 'R', 'RA', 'Inn', 'Win', 'Loss', 'Save', 'Time', 'Streak'])
# df.head()

There are two specific row-by-row cases that need to be accounted for:

 First, by default Baseball Reference repeats its column headers on tables at each new month to make it easier to view the data in the browser. Those rows each need to be filtered out.
 
 Second is strange issues coming from teams with cancelled/relocated seasons. For example, the 2025 Athletics played [this schedule](https://www.baseball-reference.com/teams/ATH/2025-schedule-scores.shtml), but were originally scheduled to play [this schedule](https://www.baseball-reference.com/teams/OAK/2025-schedule-scores.shtml) before they relocated from Oakland to Sacramento [on their way to Las Vegas](https://en.wikipedia.org/wiki/Oakland_Athletics_relocation_to_Las_Vegas).
 
 Luckily, we can just filter out both of these types of rows by only selecting for rows where `Unnamed: 2` is "boxscore", representing a game which was played and completed.

In [154]:
df = df[df['Unnamed: 2'] == 'boxscore'].drop(columns=['Unnamed: 2'])
# df.head()

Next, there are a lot of duplicate rows in the data set. Because of the way the data was pulled, an entry is made for both teams for each game. For example:

| Date | Tm | Unnamed: 4 | Opp | ... |
| ---- | -- | ---------- | --- | --- |
| Tuesday, Apr 4 | ARI | NaN | PHI | ... |
| Tuesday, Apr 4 | PHI | @ | ARI | ... |

For the Philidelphia Phillies @ Arizona Diamondbacks game on April 4, 2000.

Fixing this requires only looking at rows where `Unnamed: 4` is `NaN`, basically just selecting for home games, based on `Tm`:

In [155]:
df = df[df['Unnamed: 4'].isna()].drop(columns=['Unnamed: 4'])
# df.head()

At this point, the average number of games each team plays should be approximately 81. Sometimes this number will vary, based on cancelled games or relocated games, but in a typical season this will be exactly 81, like in 2025:

In [156]:
print(f'Home games per team: {len(df[df['year'] == 2025]) / 30}')

Home games per team: 81.0


Next, the `Date` field should be stored as a pandas datetime field for graphing purposes later:

In [157]:
# strip any double header information from the raw date string (example: Thursday, Apr 4 (1))
df['Date'] = df['Date'].str.replace(r'\s*\(\d+\)$', '', regex=True)
df['date'] = pd.to_datetime(df['Date'] + ' ' + df['year'].astype(str), format='%A, %b %d %Y')
df = df.drop(columns=['Date', 'year'])
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,date
0,ARI,CHC,5,2.0,N,49070,.92,NaN,2025-03-27
1,ARI,CHC,4,2.0,N,37449,.88,NaN,2025-03-28
2,ARI,CHC,5,3.0,N,36407,.89,NaN,2025-03-29
3,ARI,CHC,4,2.5,D,39145,.88,NaN,2025-03-30
11,ARI,BAL,4,4.0,N,20333,.83,NaN,2025-04-07


The GB column can also have some mixed data types since the raw data has values like "up 0.5" and "Tied". Also, since these will be graphed, let's convert it into a more graph friendly format; "up 0.5" becomes `0.5`, "Tied" becomes `0`, and 2.5 (two and a half back) becomes `-2.5`.

In [158]:
import re

def parse_gb(row):
    row = str(row).strip()
    # tied with lead = 0
    if row.lower() == 'tied':
        return 0.0
    # some GB columns come through as up10.5 instead of up 10.5, regex the games out
    match = re.match(r'up\s*([\d.]+)', row, re.IGNORECASE)
    if match:
        return float(match.group(1))
    
    return -float(row)

df['GB'] = df['GB'].apply(parse_gb)
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,date
0,ARI,CHC,5,-2.0,N,49070,.92,NaN,2025-03-27
1,ARI,CHC,4,-2.0,N,37449,.88,NaN,2025-03-28
2,ARI,CHC,5,-3.0,N,36407,.89,NaN,2025-03-29
3,ARI,CHC,4,-2.5,D,39145,.88,NaN,2025-03-30
11,ARI,BAL,4,-4.0,N,20333,.83,NaN,2025-04-07


There's one last scenario to account for: games with no attendance. This can happen for several reasons, but most notably occured during the COVID-19 pandemic. Filtering out rows with no attendance will solve that:

In [159]:
df = df[df['Attendance'].notna()]
# df.head()

Lastly, renaming the columns to something more readable and correcting the data types will be helpful:

In [ ]:
df = df.rename(columns={
    'Tm': 'home_team',
    'Opp': 'away_team',
    'Rank': 'rank',
    'GB': 'standings_gap',
    'D/N': 'date_night',
    'Attendance': 'attendance',
    'Orig. Scheduled': 'orig_scheduled'
})

df = df.astype({
    'rank': 'int8',
    'attendance': 'int32',
    'cLI': 'float32'
})

# df.head()

,home_team,away_team,rank,standings_gap,date_night,attendance,cLI,orig_scheduled,date
0,ARI,CHC,5,-2.0,N,49070,0.92,NaN,2025-03-27
1,ARI,CHC,4,-2.0,N,37449,0.88,NaN,2025-03-28
2,ARI,CHC,5,-3.0,N,36407,0.89,NaN,2025-03-29
3,ARI,CHC,4,-2.5,D,39145,0.88,NaN,2025-03-30
11,ARI,BAL,4,-4.0,N,20333,0.83,NaN,2025-04-07


Lastly, let's save the processed dataframe to refer to later.

In [ ]:
df.to_csv('processed_data.csv')